In [1]:
import napari
import numpy as np
from aicsimageio import AICSImage

import pandas as pd
from skimage import measure


import tifffile as tf
import matplotlib.pyplot as plt
import xarray as xr
from tifffile import tifffile
import tifftools
import os
from matplotlib import pyplot as plt
from os.path import sep
from skimage import io
from PIL import Image
import imageio


from skimage.measure import regionprops_table

import seaborn as sns

import czifile
from czifile import CziFile

import math
import matplotlib.colors as mcolors

/Users/fisherguest/miniconda3/envs/sansachen_czi/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [2]:
# this shows how many delta7 are there in each subtype

delta7_naming = pd.read_csv('neuprint_delta7_names.csv')
#delta7_naming = delta7_naming.drop(columns=["notes"])
delta7_naming['instance'] = delta7_naming['instance'].str.replace(r'^Delta7\(PB15\)_', '', regex=True)
delta7_naming.sort_values('instance', inplace=True)
delta7_naming.reset_index(drop=True, inplace=True)
subtype_summary = delta7_naming.groupby("instance").size().reset_index(name="count")
subtype_summary

,instance,count
0,L1L9R8_R,5
1,L2R7_R,5
2,L3R6_R,4
3,L4R5_R,5
4,L4R6_R,2
5,L5R4_L,5
6,L6R3_L,5
7,L6R4_L,2
8,L7R2_L,3
9,L7R3_L,1


In [3]:
fly1 = '/Users/fisherguest/Documents/sansa images/08142025/SC-0814-fly1.czi'
fly2 = '/Users/fisherguest/Documents/sansa images/08142025/SC-0814-fly2.czi'
#fly_100 = '/Users/fisherguest/Documents/sansa images/08152025/SC-0815-fly?.czi'
fly3_left = '/Users/fisherguest/Documents/sansa images/08202025/SC-0820-fly3-left.czi'
fly3_right = '/Users/fisherguest/Documents/sansa images/08202025/SC-0820-fly3-right.czi'
fly4 = '/Users/fisherguest/Documents/sansa images/08242025/SC-0824-fly4.czi'
fly5_down = '/Users/fisherguest/Documents/sansa images/08242025/SC-0824-fly5-down.czi'
fly5_up = '/Users/fisherguest/Documents/sansa images/08242025/SC-0824-fly5-up.czi'
fly6 = '/Users/fisherguest/Documents/sansa images/08242025/SC-0824-fly6.czi'
fly7 = '/Users/fisherguest/Documents/sansa images/08242025/SC-0824-fly7.czi'



In [67]:
# import image:
img = AICSImage(fly6)

# Get the data in (C, Z, Y, X)
# C: Channels (e.g., the different lasers/fluorophores); Z: Z-stacks (slices in depth); Y and X: The spatial dimensions (height and width)
# i don't have S or T.
data = img.get_image_data("CZYX", S=0, T=0)

# checked using "previous TQ method": (633 is the first channel date[0]), 568 is first (no actually second) channel, 488 is second (no actually third) channel.
red_channel   = data[1]
green_channel = data[2]

In [ ]:
# open image in napari to draw labels:
viewer = napari.Viewer()

# Add the red channel
viewer.add_image(
    red_channel, 
    name='568 Channel',
    colormap='red',
    contrast_limits=(0, 5000), # adjust this if needed
    blending='additive'
)
# Add the green channel
viewer.add_image(
    green_channel, 
    name='488 Channel',
    colormap='green',
    #contrast_limits=(50, 800), # adjust this if needed
    blending='additive'
)

napari.run()

2025-09-04 11:21:17.991 python[11088:117612141] +[IMKClient subclass]: chose IMKClient_Legacy
2025-09-04 11:21:17.991 python[11088:117612141] +[IMKInputSession subclass]: chose IMKInputSession_Legacy
2025-09-04 11:27:25.967 python[11088:117612141] The class 'NSSavePanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


In [68]:
# File paths
label_file = '/Users/fisherguest/Documents/sansa images/08242025/roi/fly6.tif'


# Load the label mask (the TIFF file you saved from napari)
label_mask = tf.imread(label_file)
# for the label tif file obtained in previous TQ method, need the below precessing:
#squeezed_mask = np.squeeze(label_mask) # so change from shape: (1, 2, Z, Y, X) to (2, Z, Y, X)
#squeezed_mask_green = squeezed_mask[1] # so only extract the green (second) channel.
print("Label mask shape:", label_mask.shape)
print("Green channel shape:", green_channel.shape)

# Check that the dimensions match (e.g., both are 3D)
if green_channel.shape != label_mask.shape:
    raise ValueError("The dimensions of the green channel and the label mask do not match. "
                     "Please verify that your label mask corresponds to the correct z, y, and x dimensions.")

Label mask shape: (131, 1839, 1839)
Green channel shape: (131, 1839, 1839)


In [69]:
# check which z stack I drew the label on.
sums = label_mask.sum(axis=(1,2))
painted_slices = np.where(sums > 0)[0]
print("You painted on slice(s):", painted_slices)

You painted on slice(s): [65]


In [70]:
z_ranges = [
(12, 81),
(20, 83),
(21, 97),
(31, 98),
(24, 102),
(35, 96),
(43, 109),
(48, 123),
(61, 126),
(43, 121),
(35, 110),
(33, 94),
(31, 90),
(33, 89),
(44, 93),
(22, 93),
(5, 75),
(22, 70),
]

In [87]:
# variable input cell
red_threshold_lower = 500
green_threshold_lower = 250

In [88]:
# --- compute stats with proper red-first, green-subset logic ---

n_glom = int(label_mask.max())

mean_red    = np.zeros(n_glom, dtype=float)
mean_green  = np.zeros(n_glom, dtype=float)
total_red   = np.zeros(n_glom, dtype=float)
total_green = np.zeros(n_glom, dtype=float)
n_red_px    = np.zeros(n_glom, dtype=int)
n_green_px  = np.zeros(n_glom, dtype=int)

# Optional: basic validation of z_ranges length
if len(z_ranges) != n_glom:
    raise ValueError(f"z_ranges has length {len(z_ranges)} but label_mask has {n_glom} labels.")

for i in range(1, n_glom + 1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)          # THIS COMMENT WILL SOLVE MANY POTENTIAL QUESTIONS: label_mask size is in this format: (Z, Y, X)
    if zs.size == 0:
        # No pixels for this label; keep NaNs/zeros
        continue
    z_draw = zs[0]                                  # AICSImage makes the image format in this way! (i don't know why not in X->Y->Z).  z_draw = z layer that you draw labels on (a number).

    # 2) 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)               # roi2d = a T/F information on that one z plane you have label 1 draw onto.
                                                    # label_mask[z_draw] gives 2d array (Y, X): entries are the integer labels (background:0, ROIs:1-18) (the entries are numbers because that's what the label_mask tiff is like).
                                                    # label_mask[z_draw] == i: give each pixel True of False based on if it's has label 1. So now on this 2D sheet, only the pixels you draw (eg.) label 1 on has a T value.

    # 3) build the Z-range selector:
    z0, z1 = z_ranges[i - 1]                        # z0=a lowest z layer that the current label reaches, while z1=the highesr z laye the current label goes to.
    Z, Y, X = label_mask.shape                      # label_mask is the full X * full Y * full Z that your current image (eg. fly1) has. Z, Y, X are the length of the image (number of pixels).
    mask_z = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)    # NOT USED IN BELOW PARTS!!!
                                                            # np.arange(Z): generates the array [0, 1, 2, …, Z−1] (yes, the entries are numbers, not bool) --> this mask_z is only a T/F filter for the first dimension (which is Z).
                                                            # np.arange(Z) >= z0: generates the array [False, F, ...T, T, ... F]
                                                            # () & (): generates the array with True only for indices between z0 and z1.
                                                            
    # 4) 3D ROI over the Z-range by broadcasting roi2d across selected Z
    # This is equivalent to your earlier construction, but a bit more explicit:
    roi = np.zeros_like(label_mask, dtype=bool)             # roi = 3D array of F as the only entries, the size = the label_mask (or the size of your image).
    roi[z0:z1+1, :, :] = roi2d[None, :, :]                  # roi[z0:z1+1, :, :] = a 3D array of bool, but only with the z slices in the range of this label 1. ":" means "no slicing and include all".
                                                            # roi2d[None, :, :]  = the previous 2D array of T/F now added one more dimension (Z). chatgpt said the new Z dimension is "a new axis".
                                                            # roi[z0:z1+1, :, :] = roi2d[None, :, :]: broadcasts (1, Y, X) → (z0:z1+1, Y, X).
                                                            # NOW, roi is the 3D array of T/F which basically copy and paste roi2d to all the z slices in wherez0:z1+1. (on each z slice: T in where the label is drawn on, and definitely F outside the labels.)
                                                                # yes, weird grammar that i have never heard before (called "broadcast").

    # Slice channels to the Z-range (faster & clearer)
    red_sub   = red_channel[z0:z1+1]                        # shape (Zsel, Y, X): red_channel[z0:z1+1]: takes only the z-stacks from the original image (red channel only) that are within the *current label's* z-stack range. No changes were made to the red and green channels. 
                                                            # red_sub = entries are fluorescence values, not T/F!!!                                   
    green_sub = green_channel[z0:z1+1]                      # again, only select out the correct z stacks, haven't picked the inner pixels yet!
    roi_sub   = roi[z0:z1+1]                                # roi[z0:z1+1]: only get the z0:z1+1 z slices in that 3D T/F array you just built.
                                                            # roi_sub = 3D!

    # ---- Thresholding logic ----
    # A) Pixels inside ROI (any Z in the chosen range)
    mask_roi = roi_sub
    # B) Finalized red pixels (apply red threshold INSIDE ROI)
    mask_red = mask_roi & (red_sub > red_threshold_lower)           # mask_red = a 3D array of T/F where T is in pixels only both conditions are met.
    # C) Green subset pixels (subset of red pixels that also pass green threshold)
    mask_green = mask_red & (green_sub > green_threshold_lower)     # mask_green = a 3D array of T/F where T is in pixels only both conditions are met. NOTE: mask_green is built on mask_red!!!!!!!!!

    # ---- Extract values ----
    red_vals   = red_sub[mask_red]                                  # applies the True/False mask and keeps only the voxels where the mask is T to red_sub, which is the origianl image with only some z slices included. red_vals is no longer a 3D array, but a 1D array, with entries being the signal value from the T pixels.
    green_vals = green_sub[mask_green]

    # ---- Save stats (NaN if empty) ----
    n_red_px[i - 1]   = red_vals.size
    n_green_px[i - 1] = green_vals.size

    mean_red[i - 1]    = red_vals.mean()   if red_vals.size   else np.nan
    total_red[i - 1]   = red_vals.sum()    if red_vals.size   else np.nan
    mean_green[i - 1]  = green_vals.mean() if green_vals.size else np.nan
    total_green[i - 1] = green_vals.sum()  if green_vals.size else np.nan

# Optional: quick sanity check that subset property holds
if not np.all(n_green_px <= n_red_px):
    print("Warning: some ROIs have more green pixels than red pixels — check thresholds and masks.")

# Assemble results
df_means = pd.DataFrame(
    {
        'mean_red':    mean_red,
        'mean_green':  mean_green,
        'total_red':   total_red,
        'total_green': total_green,
        'n_red_px':    n_red_px,
        'n_green_px':  n_green_px,
    },
    index=np.arange(1, n_glom + 1)
)
df_means

,mean_red,mean_green,total_red,total_green,n_red_px,n_green_px
1,2635.865872,598.191929,1.167855e+09,99866348.0,443063,166947
2,1623.318193,508.071756,4.687672e+08,18274833.0,288771,35969
3,1796.798288,572.655751,7.174903e+08,54540879.0,399316,95242
4,1534.879839,545.357097,2.886311e+08,13601206.0,188048,24940
5,1732.517472,554.283139,6.517835e+08,39657850.0,376206,71548
6,1473.927459,553.659168,3.297515e+08,23651766.0,223723,42719
7,1488.229869,652.238606,2.074548e+08,10103176.0,139397,15490
8,1251.417127,402.140019,1.494405e+08,2490051.0,119417,6192
9,1896.032858,485.707260,1.226894e+09,70259498.0,647085,144654
10,1648.203626,556.535277,2.905602e+08,5498012.0,176289,9879


In [59]:
df_means = pd.DataFrame(
    [mean_red, mean_green, total_red, total_green],
    index=['mean_red', 'mean_green', 'total_red', 'total_green']
)

df_means.T.to_csv(f"fly7-l3r6.csv")

In [34]:
# this cell is for when one brain is split into 2 images, and need to have the values combined into one csv...

# build the dataframe
data = {
    "mean_red": [
        95.689903, 244.708255, 85.982069, 87.822602, 93.642687, 106.009298,
        128.30713, 119.484986, 97.095008, 234.183041, 105.323247, 106.027822,
        126.8595, 141.105013, 153.156412, 205.923968, 146.704492, 239.888989
    ],
    "mean_green": [
        22.3198, 24.493158, 17.833971, 13.558714, 14.336447, 17.595527,
        20.401557, 19.864207, 14.987217, 26.001336, 21.642499, 23.489537,
        24.551414, 26.666706, 29.18899, 35.560629, 36.782444, 40.622732
    ],
    "total_red": [
        120096323, 613873783, 95379737, 84603465, 107275845, 135048107,
        208664859, 154102773, 100665483, 662037565, 128691526, 123137319,
        205863156, 268482061, 176626254, 311569965, 278433829, 535220205
    ],
    "total_green": [
        23466471, 68059125, 19783188, 13061719, 16423647, 22415417,
        33178889, 25619365, 15538342, 73506012, 26444363, 27279997,
        39841176, 50739034, 33661940, 53804440, 69810246, 97960443
    ]
}

fly3_df = pd.DataFrame(data)

# save it as fly3.csv in current directory
fly3_df.to_csv("fly5.csv", index=True)

In [86]:
# --- sanity checks ---
Z, Y, X = red_channel.shape
assert green_channel.shape == (Z, Y, X)
assert label_mask.shape   == (Z, Y, X)
n_glom = int(label_mask.max())
assert len(z_ranges) == n_glom, f"Need one (z0, z1) per ROI (got {len(z_ranges)} for {n_glom})."

# --- outputs ---
mask_red_labels = np.zeros((Z, Y, X), dtype=np.int32)   # 0 = background; i = label i (colored by napari automatically)
mask_green_full = np.zeros((Z, Y, X), dtype=bool)       # binary (we'll show as green image)

for i in range(1, n_glom + 1):
    # find a painted slice for ROI i
    zs = np.where(label_mask == i)[0]
    if zs.size == 0:
        continue
    z_draw = zs[0]
    roi2d = (label_mask[z_draw] == i)                   # (Y, X) bool

    # z-range for this ROI
    z0, z1 = z_ranges[i - 1]
    zlen = z1 - z0 + 1

    # broadcast 2D ROI across its z-range
    roi_sub = np.repeat(roi2d[None, :, :], zlen, axis=0)   # (zlen, Y, X)

    # slice channels to that range
    red_sub   = red_channel[z0:z1+1]                    # (zlen, Y, X)
    green_sub = green_channel[z0:z1+1]                  # (zlen, Y, X)

    # thresholds: green subset ⊆ red
    mask_red_sub   = roi_sub & (red_sub   > red_threshold_lower)
    mask_green_sub = mask_red_sub & (green_sub > green_threshold_lower)

    # paint red-passing voxels with this ROI's integer ID (i) into the label volume
    z_rel, y_idx, x_idx = np.where(mask_red_sub)        # indices relative to z0
    mask_red_labels[z0 + z_rel, y_idx, x_idx] = i

    # accumulate green subset as binary
    mask_green_full[z0:z1+1] |= mask_green_sub

# --- visualize in napari ---
viewer = napari.Viewer()

# raw channels
viewer.add_image(red_channel,   name='Red (raw)',   colormap='red', contrast_limits=(0, 5000), blending='additive')
viewer.add_image(green_channel, name='Green (raw)', colormap='green',   blending='additive')

# mask_red as LABELS → default napari per-label coloring (each ROI a different color)
viewer.add_labels(mask_red_labels, name='Mask – red per ROI', opacity=0.6)

# mask_green as IMAGE → solid green binary overlay
viewer.add_image(mask_green_full.astype(np.float32), name='Mask – green subset (binary)',
                 colormap='red', blending='additive', opacity=0.7)

# optionally also show your original ROI labels
# viewer.add_labels(label_mask.astype(np.int32), name='ROI labels (drawn)', opacity=0.25)

napari.run()

In [80]:
# this cell works previously, when i only want to see what mask_red labels look like on top of the image. (no mask_green!!!)

# check the mask in napari
propagated_labels = np.zeros_like(label_mask, dtype=np.int32)  # shape (Z, Y, X)
# Your existing loop, with one extra line to “paint” into propagated_labels
for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]
    # 2) 2D ROI on drawn slice
    roi2d = (label_mask[z_draw] == i)
    # 3) Z‑range mask
    z0, z1 = z_ranges[i-1]
    Z, Y, X = label_mask.shape
    mask_z = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)
    # 4) broadcast to 3D
    roi = mask_z[:, None, None] & roi2d[None, :, :]
    # 5) your threshold step and mean extraction
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]
    mask_thr  = (red_sub > red_threshold_lower)
    # … compute mean_red[i-1], mean_green[i-1] as before …
    # — NEW: paint this ROI into your propagated label map —
    propagated_labels[roi & (red_channel > red_threshold_lower)] = i
# Save the new propagated label volume to TIFF
tf.imwrite('propagated_glom_labels.tif', propagated_labels.astype(np.uint16))
# Now open it in Napari alongside your images
viewer = napari.Viewer()
viewer.add_image(red_channel,   name='568 nm', colormap='red',   blending='additive')
viewer.add_image(green_channel, name='488 nm', colormap='green', blending='additive')
viewer.add_labels(propagated_labels, name='Propagated ROIs', opacity=0.6)
napari.run()



In [ ]:
# this cell definitely works, but failed to apply the green_threshold_lower...

n_glom    = int(label_mask.max())  # should be 18
mean_red   = np.zeros(n_glom, dtype=float)
mean_green = np.zeros(n_glom, dtype=float)
total_red   = np.zeros(n_glom, dtype=float)
total_green = np.zeros(n_glom, dtype=float)
for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)  # THIS COMMENT WILL SOLVE MANY POTENTIAL QUESTIONS: label_mask size is in this format: (Z, Y, X)
    z_draw = zs[0]                          # AICSImage makes the image format in this way! (i don't know why not in X->Y->Z). This makes z_draw the z layer that you draw labels on.
    # 2) extract the 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)   # label_mask[z_draw] gives 2d array (Y, X): entries are the integer labels (background:0, ROIs:1-18)
                                        # label_mask[z_draw] == i: give each pixel True of False based on if it's has label 1. So now on this 2D sheet, only the pixels you draw (eg.) label 1 on has a T value.
    # 3) build a 1D mask for your Z‑range:
    z0, z1   = z_ranges[i-1]            # z0=a lowest z layer that the current label reaches, while z1=the highesr z laye the current label goes to.
    Z, Y, X  = label_mask.shape         # label_mask is the full X * full Y * full Z that your current image (eg. fly1) has.
    mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # np.arange(Z): generates the array [0, 1, 2, …, Z−1]
                                                            # np.arange(Z) >= z0: generates the array [False, F, ...T, T, ... F]
                                                            # () & (): generates the array with True only for indices between z0 and z1.
    # 4) make the final 3d mask:
    roi = mask_z[:, None, None] & roi2d[None, :, :]         # mask_z[:, None, None]: mask_z is originally 1d array, but using None it adds on dimension with length of 1
                                                                # so now it's a 3d array with the first dimension (or first element) length of z-stack range, the second and the third length of 1
                                                            # mask_z[:, None, None] size: (z-stack range, 1, 1)
                                                            # roi2d[None, :, :] size: (1, 1863, 1863) with some x and some y True.
                                                            # Then, the "&" step: for example:
                                                                    # mz goes from (3,1,1) → (3,2,2) by repeating the single 1×1 mask across the 2×2 grid in X and Y.
                                                                    # r2 goes from (1,2,2) → (3,2,2) by repeating the same 2×2 slice across the 3 Z‑layers.
                                                            # in this new 3d array: first dimension (element) is Z!
                                                            # so: in this new 3d array:
                                                                    # first look at mask_z, if it is true, that means there is label in this z-stack,
                                                                        # and thus i will allow the Y and X info from roi2d to get copied into this Z dimension.
                                                                    # if it's False, then there is no label on this z-stack,
                                                                        # so i won't allow roi2d to copy anything to here, and this whole Z dimension will have all False values
            # for example: mask_z = np.array([ True, False,  True ]), roi2d  = np.array([[ True, False], [False,  True]]),
                # then roi = array([[[ True, False], [False,  True]],   # z=0 slice uses roi2d because mask_z[0] is True
                #                   [[False, False], [False, False]],   # z=1 slice is all False because mask_z[1] is False
                #                   [[ True, False], [False,  True]]])  # z=2 slice repeats roi2d because mask_z[2] is True.
    # — b) zero out outside your propagated ROI —
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]      # red_channel[z0:z1+1]: takes only the z-stacks from the original image (red channel only) that are within the current label z-stack range
                                                           # roi[z0:z1+1]: only slicing in the first (Z) dimension!
                                                           # after multiplication: it's a smaller Z * 1863 * 1863 3d array, with the in-label pixels having value of 1*it's original value (because roi[z0:z1+1] has T or F entries, so multiplying a T=*1 while multiplying a F=*0)
    green_sub = green_channel[z0:z1+1]                     # only select out the correct z stacks, haven't picked the inner pixels yet!
    # — c) threshold & extract means —
    mask      = (red_sub > red_threshold_lower)            # after making the correct label in each z stack, "mask" is the mask for correct pixels.
    red_vals   = red_sub  [mask]                           # only extract the value from the pixel in "mask"
    green_vals = green_sub[mask]
    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan
    total_red[i-1]   = red_vals.sum()   if red_vals.size   else np.nan
    total_green[i-1] = green_vals.sum() if green_vals.size else np.nan


In [ ]:
# don't know why i have this cell...but i will just keep it here...



props_green = regionprops_table(
    label_mask, 
    intensity_image=green_channel,
    properties=('label', 'area', 'mean_intensity', 'max_intensity', 'min_intensity')
)
df_green = pd.DataFrame(props_green)
# Rename columns to indicate green measurements
df_green = df_green.rename(columns={
    'label': 'label',
    'area': 'area',
    'mean_intensity': 'green_mean_intensity',
    'max_intensity': 'green_max_intensity',
    'min_intensity': 'green_min_intensity'
})

props_red = regionprops_table(
    label_mask, 
    intensity_image=red_channel,
    properties=('mean_intensity', 'max_intensity', 'min_intensity')
)
df_red = pd.DataFrame(props_red)
# Rename columns to indicate red measurements
df_red = df_red.rename(columns={
    'mean_intensity': 'red_mean_intensity',
    'max_intensity': 'red_max_intensity',
    'min_intensity': 'red_min_intensity'
})

df_combined = pd.concat([df_green, df_red], axis=1)
df_combined

,label,area,green_mean_intensity,green_max_intensity,green_min_intensity,red_mean_intensity,red_max_intensity,red_min_intensity
0,1,25892.0,26.886258,150.0,3.0,89.018925,1286.0,0.0
1,2,30601.0,34.594000,427.0,4.0,228.817816,7016.0,0.0
2,3,38668.0,46.797352,504.0,7.0,314.064498,6968.0,0.0
3,4,34910.0,41.592237,304.0,5.0,257.410771,7312.0,0.0
4,5,42942.0,50.595547,1168.0,4.0,241.474454,6336.0,0.0
5,6,39097.0,47.902371,402.0,5.0,218.569225,9609.0,0.0
6,7,42310.0,93.945805,960.0,7.0,1647.427393,13304.0,0.0
7,8,47045.0,35.267829,329.0,5.0,123.847316,3310.0,0.0
8,9,36320.0,26.517098,259.0,2.0,55.211151,1109.0,0.0
9,10,34457.0,35.726442,156.0,2.0,82.081261,1029.0,0.0
